# Armar Panel de datos
## Objetivo
- Armar el panel de datos con las bases que tengo identificadas


# Herramientas

## Paquetes

In [1]:
# Manejar datos
import pandas as pd
pd.set_option('display.float_format', lambda x: '{:,.2f}'.format(x)) # Configuración para mostrar separador de miles

# Manejar datos espaciales
import geopandas as gpd
from shapely.validation import make_valid

# Calculo numérico
import numpy as np

# Utilidades
from tqdm import tqdm # revisar avance de ciclos
import warnings # Manejar warnings
import unicodedata # Manejar caracteres especiales de los strings

# Gráficas
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

# Procesar datos

## Panel CEDE
La información de la UPRA en el CEDE está mal procesada. Por eso decidí procesar directamente la información de la UPRA-EVA

```python
CEDE_agricultura = pd.read_stata('../../Data/0_Panel_CEDE/Microdatos/PANEL_AGRICULTURA_Y_TIERRA(2024).dta',
                                preserve_dtypes=True).copy()

# Reparar variables 
CEDE_agricultura['CODIGO_MUNICIPIO'] = CEDE_agricultura['codmpio'].astype("Int64").astype(str).str.zfill(5)
CEDE_agricultura['ANNO'] = CEDE_agricultura['ano'].astype("Int64")

# Desfragmentar el DF
CEDE_agricultura = CEDE_agricultura.copy()

# Conservar solo las variables de interes
vbles_aguacate = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('aguacate')].to_list()
vbles_cafe = CEDE_agricultura.columns[CEDE_agricultura.columns.str.contains('cafe')].to_list()
vbles_id = ['codmpio', 'CODIGO_MUNICIPIO', 'ANNO']
vbles_interes = vbles_id + vbles_aguacate + vbles_cafe
CEDE_agricultura = CEDE_agricultura[vbles_interes]


# Organizar datos en formato long
panel_agricultura = CEDE_agricultura.melt(
    id_vars=vbles_id,
    value_vars=vbles_resultado,
    value_name='valor'
)

# Eliminar observaciones de NaN
print(f"Número de obs ANTES de eliminar NAN {len(panel_agricultura)}")
panel_agricultura = panel_agricultura.dropna(subset='valor')
print(f"Número de obs DESPUES de eliminar NAN {len(panel_agricultura)}")


mascara = panel_agricultura_info.columns[panel_agricultura_info.columns.str.contains('aguacate')]
display(panel_agricultura_info[mascara])
# El panel CEDE no identifica bien las variedades de aguacate
# En 2019, no tiene registros de "Aguacate" pero es el único año en que tiene registros de "aguacatenhnp"
# Por eso, voy a consultar directamente los datos de la UPRA
```

## UPRA EVA 2024

In [2]:
# Cargar datos
UPRA = pd.read_excel('../../Data/9_UPRA_EVA/20250617_BaseAgricola20192024.xlsx',
                         sheet_name='BasePagina',
                         skiprows=7,
                         dtype={'Código Dane municipio':str, 'Año':int})

# Mientras procesaba los datos pensé que los demás cultivos compiten directamente por
# factores los factores de producción para los cultivos. Por eso voy a cargar el area sembrada
# y cosechada de los demas cultivos. Voy a separar por grupo de cultivo
vbles_conservar = [
    'Código Dane municipio',
    'Desagregación cultivo',
    'Grupo cultivo',
    'Año',
    'Área sembrada (ha)',
    'Área cosechada (ha)',
    'Producción (t)'
]

# Mostrar los cultivos disponibles en la base de datos
UPRA['Cultivo'].unique()

array(['Aguacate', 'Ahuyama', 'Ají', 'Ajo', 'Apio', 'Arveja', 'Brevo',
       'Brócoli', 'Café', 'Caña', 'Cebolla de bulbo', 'Cebolla de rama',
       'Cilantro', 'Col', 'Coliflor', 'Espinaca', 'Fresa', 'Frijol',
       'Guayaba', 'Haba', 'Habichuela', 'Lechuga', 'Lima', 'Limón',
       'Mandarina', 'Mango', 'Mora', 'Naranja', 'Otras hortalizas',
       'Papa', 'Pepino Cohombro', 'Perejil', 'Pimentón', 'Plátano',
       'Rábano', 'Remolacha', 'Repollo', 'Tomate de árbol', 'Tomate',
       'Yuca', 'Zanahoria', 'Banano', 'Cacao', 'Lulo', 'Maíz',
       'Granadilla', 'Gulupa o cholupa', 'Fique', 'Arroz', 'Maracuyá',
       'Piña', 'Plantas aromáticas', 'Uchuva', 'Calabacín, calabaza',
       'Limonaria', 'Mamoncillo', 'Moringa', 'Papaya', 'Tamarindo',
       'Guanábana', 'Coco', 'Ñame', 'Tabaco', 'Arracacha', 'Sábila',
       'Sacha inchi', 'Stevia', 'Zapote', 'Cardamomo', 'Palma de aceite',
       'Agraz - mortiño', 'Patilla', 'Pitahaya', 'Arándano', 'Acelga',
       'Tangelo', 'Romero',

In [3]:
UPRA.head()

,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Desagregación cultivo,Cultivo,Ciclo del cultivo,Grupo cultivo,Subgrupo,Año,Periodo,Área sembrada (ha),Área cosechada (ha),Producción (t),Rendimiento (t/ha),Nombre científico del cultivo,Código del cultivo,Estado físico del cultivo
0,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2019,2019,24.00,23.00,138.00,6.00,Persea americana,2040299,En fresco
1,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2020,2020,8.52,3.52,21.12,6.00,Persea americana,2040299,En fresco
2,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2021,2021,8.52,4.52,27.12,6.00,Persea americana,2040299,En fresco
3,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2022,2022,17.17,8.52,51.12,6.00,Persea americana,2040299,En fresco
4,5,Antioquia,05001,Medellín,Aguacate demás variedades,Aguacate,Permanente,Frutales,Demás frutales,2023,2023,14.97,8.52,51.12,6.00,Persea americana,2040299,En fresco


### Extraer información de un cultivo dado

```python
# Este es el mismo código que uso para extraer información de multiples cultivos solo que puntualmente para el Agucate.
# Lo dejo acá por si me sirve en el futuro para entender rápidamente con lo puntual y con lo general
# Saludos desde el 15 de abril de 2026



# Extraer datos del aguacate -----------------------------
filas_aguacate = UPRA['Cultivo']=='Aguacate'
UPRA_aguacate = UPRA.loc[filas_aguacate, vbles_conservar]

# Calcular area de cultivos diferentes al aguacate
UPRA_aguacate = UPRA_aguacate.rename(columns={'Desagregación cultivo':'cultivo'})
UPRA_aguacate['Grupo cultivo'] = UPRA_aguacate['cultivo']

# Organizar en formato long
UPRA_aguacate = UPRA_aguacate.melt(
    id_vars=['Código Dane municipio', 'Año', 'cultivo', 'Grupo cultivo'],
    value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
    var_name='variable',
    value_name='valor'
)

# Calcular la suma de las variables de resultado para los demás cultivos
vbles_resultado = ['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)']
UPRA_noAguacate = UPRA.loc[~filas_aguacate].groupby(['Código Dane municipio', 'Año', 'Grupo cultivo', ])[vbles_resultado].sum()
UPRA_noAguacate = UPRA_noAguacate.reset_index()
UPRA_noAguacate['cultivo'] = 'No aguacate'

# Organizar en formato long
UPRA_noAguacate = UPRA_noAguacate.melt(
    id_vars=['Código Dane municipio', 'Año', 'cultivo', 'Grupo cultivo'],
    value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
    var_name='variable',
    value_name='valor'
)
```

### Extraer información de multiples cultivos
Para cada `cultivo` de la lista `cultivos`, extraigo y organizo
1. Área Sembrada del cultivo de todos los demás cultivos (suma)
1. Área Cosechada del cultivo de todos los demás cultivos (suma)
1. Producción del cultivo de todos los demás cultivos (suma)

In [4]:
UPRA['Grupo cultivo']

0                    Frutales
1                    Frutales
2                    Frutales
3                    Frutales
4                    Frutales
                 ...         
141068    Raíces y tubérculos
141069    Raíces y tubérculos
141070    Raíces y tubérculos
141071    Raíces y tubérculos
141072    Raíces y tubérculos
Name: Grupo cultivo, Length: 141073, dtype: object

In [5]:
# Lista con el nombre de los cultivos de interés (ver lista de la celda de arriba)
Cultivos = ['Aguacate', 'Café']

# DataFrame vacio para almacenar resultados
panel_cultivos = pd.DataFrame()

for cultivo_interes in Cultivos:

    # Extraer datos del cultivo -----------------------------
    filas_cultivo_interes = UPRA['Cultivo']==cultivo_interes
    UPRA_cultivo_interes = UPRA.loc[filas_cultivo_interes, vbles_conservar]

    # Calcular area de cultivos diferentes al cultivo en cuestión
    UPRA_cultivo_interes = UPRA_cultivo_interes.rename(columns={'Desagregación cultivo':'cultivo'})
    #UPRA_cultivo_interes['Grupo cultivo'] = UPRA_cultivo_interes['cultivo'] # Crear columna redundante
    # En el panel final, la columna "cultivo" va a tomar los valores "Aguacate Haas", 
    # "Aguacate Papelillo",... "No Aguacate". La última columna es la que va a identificar los valores de
    # todo los cultivos diferentes al cultivo de interés

    # Organizar en formato long
    UPRA_cultivo_interes = UPRA_cultivo_interes.melt(
        id_vars=['Código Dane municipio', 'Año', 'cultivo'],
        value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
        var_name='variable',
        value_name='valor'
    )
    UPRA_cultivo_interes['tipo_variable'] = 'Resultado' # Identificar el tipo de variable
    
    # Calcular la suma de las variables de resultado para los demás cultivos
    vbles_resultado = ['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)']
    UPRA_diferente_a_cultivo_interes = UPRA.loc[~filas_cultivo_interes].groupby(['Código Dane municipio', 'Año', 'Grupo cultivo', ])[vbles_resultado].sum()
    UPRA_diferente_a_cultivo_interes = UPRA_diferente_a_cultivo_interes.reset_index()
    UPRA_diferente_a_cultivo_interes['cultivo'] = f'No_{cultivo_interes}_' + UPRA_diferente_a_cultivo_interes['Grupo cultivo']
    UPRA_diferente_a_cultivo_interes['tipo_variable'] = 'Control - Otros cultivos'# Identificar el tipo de variable
    
    # Organizar en formato long
    UPRA_diferente_a_cultivo_interes = UPRA_diferente_a_cultivo_interes.melt(
        id_vars=['Código Dane municipio', 'Año', 'cultivo', 'tipo_variable'],
        value_vars=['Área sembrada (ha)', 'Área cosechada (ha)', 'Producción (t)'],
        var_name='variable',
        value_name='valor'
    )
    
    # Unir datos del panel
    panel_cultivo_interes = pd.concat(
        [UPRA_cultivo_interes, UPRA_diferente_a_cultivo_interes],
        axis=0,
        ignore_index=True
    )
    
    # Unir paneles de cultivos de interes
    panel_cultivos = pd.concat(
        [panel_cultivos, panel_cultivo_interes],
        axis=0,
        ignore_index=True
    )
    

In [6]:
# Para organizar los datos en el PANEL, necesito que todo quede con la misma estructura.
# Las columnas del Panel van a ser

# Codimo_mpio --- Año   --- Variable                        --- Valor  --- Auxiliar         --- Tipo_variable

# NNNN        --- 2015  --- 'Aguacate_Hass_Producción (t)'  --- 1,000  --- Producción (t) --- Resultado
# NNNN        --- 2015  --- 'No_Aguacate_Producción (t)'    --- 1,000  --- Producción (t) --- Control
# NNNN        --- 2015  --- 'Area_titulo_minero_m2'         --- 20,000 --- m2             --- Instrumento

# Organizo los nombres anteriores del DF en los nombres que necesito para armar el panel
panel_cultivos['auxiliar'] = panel_cultivos['variable']
panel_cultivos['variable'] = panel_cultivos['cultivo'] + '_' + panel_cultivos['auxiliar']

In [7]:
print(f"{'*'*20} cultivos analizados")
display(panel_cultivos['cultivo'].unique())

print(f"{'*'*20} Panel")
display(panel_cultivos.head().drop(columns={'cultivo'}))

# Explorar panel creado
print(f"\n{'*'*20} La columna 'valor' es el conteo de observaciones")
panel_cultivos.groupby(['variable'])['valor'].count().reset_index().head(20)

******************** cultivos analizados


array(['Aguacate demás variedades', 'Aguacate Hass', 'Aguacate Papelillo',
       'No_Aguacate_Cultivos tropicales tradicionales',
       'No_Aguacate_Frutales', 'No_Aguacate_Hortalizas',
       'No_Aguacate_Leguminosas', 'No_Aguacate_Raíces y tubérculos',
       'No_Aguacate_Cultivos para condimentos, bebidas medicinales y aromáticas',
       'No_Aguacate_Cereales', 'No_Aguacate_Oleaginosas', 'Café',
       'No_Café_Cultivos tropicales tradicionales', 'No_Café_Frutales',
       'No_Café_Hortalizas', 'No_Café_Leguminosas',
       'No_Café_Raíces y tubérculos',
       'No_Café_Cultivos para condimentos, bebidas medicinales y aromáticas',
       'No_Café_Cereales', 'No_Café_Oleaginosas'], dtype=object)

******************** Panel


,Código Dane municipio,Año,variable,valor,tipo_variable,auxiliar
0,05001,2019,Aguacate demás variedades_Área sembrada (ha),24.00,Resultado,Área sembrada (ha)
1,05001,2020,Aguacate demás variedades_Área sembrada (ha),8.52,Resultado,Área sembrada (ha)
2,05001,2021,Aguacate demás variedades_Área sembrada (ha),8.52,Resultado,Área sembrada (ha)
3,05001,2022,Aguacate demás variedades_Área sembrada (ha),17.17,Resultado,Área sembrada (ha)
4,05001,2023,Aguacate demás variedades_Área sembrada (ha),14.97,Resultado,Área sembrada (ha)



******************** La columna 'valor' es el conteo de observaciones


,variable,valor
0,Aguacate Hass_Producción (t),1191
1,Aguacate Hass_Área cosechada (ha),1191
2,Aguacate Hass_Área sembrada (ha),1191
3,Aguacate Papelillo_Producción (t),527
4,Aguacate Papelillo_Área cosechada (ha),527
5,Aguacate Papelillo_Área sembrada (ha),527
6,Aguacate demás variedades_Producción (t),3033
7,Aguacate demás variedades_Área cosechada (ha),3033
8,Aguacate demás variedades_Área sembrada (ha),3033
9,Café_Producción (t),3807


## Precios insumos agropecuarios - DANE SIPSA
Los datos de insumos agropecuarios del DANE están separados en tres categorías principales
1. Insumos Agrícolas 
1. Insumos Agropecuarios
1. Insumos de produccion

En general me interesan el primero y el tercero. Cada uno de esos tipos de insumos está separado en hojas diferentes. En las siguientes celdas importo, limpio y organizo los datos teniendo esto en cuenta

### Insumos agrícolas

In [8]:
# Ruta del archivo
ruta_insumos_agropecuarios = '../../Data/6_SIPSA_Insumos_Factores/anex-SIPSAInsumos-SeriesHistoricasMun-2021-2026.xlsx'

# Diccionario con la hoja de cada tipo de insumo agrícola
insumos_agricolas = {
    'Bioinsumos':'1.1',
    'Coadyuvantes, molusquicidas, reguladores fisiológicos y otros': '1.2',
    'Fertilizantes, enmiendas y acondicionadores de suelo': '1.3',
    'Fungicidas':'1.4',
    'Herbicidas':'1.5',
    'Insecticidas, acaricidas y nematicidas': '1.6'
}

# Crear DF vacío para almacenar los datos
df_insumos_agricolas = pd.DataFrame()

# Cargar todas las hojas y unirlas
for tipo_insumo, hoja_excel in tqdm(insumos_agricolas.items()):
    
    # Informar avance
    #print(f"\n{'*'*20} Procesando: {tipo_insumo}")
    
    # Cargar la hoja de cada tipo de insumo_agricola
    df_insumos = pd.read_excel(
        ruta_insumos_agropecuarios,
        sheet_name=hoja_excel,
        skiprows=8,
        dtype={'Código municipio':str}
    )
    
    # Eliminar las las notas del DANE (filas del final). Son las que tienen NAN en la 
    # columna 'Precio promedio'
    df_insumos = df_insumos.dropna(subset='Precio promedio ')
    
    # Agregar identificador del tipo de insumo agrícola
    df_insumos['tipo_insumo'] = tipo_insumo
    
    # Unir lasobservaciones en un solo df
    df_insumos_agricolas = pd.concat(
        [df_insumos_agricolas, df_insumos],
        axis=0,
        ignore_index=True
    )
    
# Eliminar columna "Unnamed: 9"
df_insumos_agricolas = df_insumos_agricolas.drop(columns='Unnamed: 9')

100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [01:07<00:00, 11.17s/it]


In [9]:
# Revisar el número de presentaciones de cada producto
df_presentaciones  = df_insumos_agricolas.groupby('Nombre del producto')['Presentación del producto'].agg([
    ('n_presentaciones', 'nunique'),
    ('presentaciones', lambda x: '; '.join(sorted(x.dropna().unique())))
]).reset_index()

df_presentaciones.sort_values('n_presentaciones', ascending=False).head(5)

,Nombre del producto,n_presentaciones,presentaciones
540,Panzer 480 Sl,5,1 litro; 10 litros; 20 litros; 250 centímetros...
240,Daconil 720 Sc,5,1 litro; 20 litros; 250 centímetros cúbicos; 4...
469,Metsulfuron Metil 600 Wg,4,1 kilogramo; 15 gramos; 20 gramos; 500 gramos
331,Fulminator 600 Ec,4,1 litro; 100 centímetros cúbicos; 250 centímet...
175,Carbendazim 500 Sc,4,1 litro; 100 centímetros cúbicos; 250 centímet...


In [10]:
# calcular precios unitarios de las diferentes presentaciones

# Separar la cantidad de la unidad
df_insumos_agricolas[["cantidad", "unidad"]] = (
    df_insumos_agricolas['Presentación del producto']
    .str.extract(r"([\d\.]+)\s*(.*)") # usar expresiones regulares para separar el dígito \d del texto
)

# Asegurarse que las cantidades son numéricas
df_insumos_agricolas["cantidad"] = df_insumos_agricolas["cantidad"].astype(float)

# homogeneizar strings de unidades
df_insumos_agricolas["unidad"] = df_insumos_agricolas["unidad"].str.lower().str.strip()

# Explorar las unidades únicas
display(df_insumos_agricolas["unidad"].unique())

# definir a qué unidad se va a convertir cada unidad
#convertir_unidades = pd.DataFrame()

array(['litro', 'gramos', 'kilogramos', 'litros', 'kilogramo',
       'centímetros cúbicos'], dtype=object)

In [11]:
# Definir factores de conversión para las unidades
factores_conversión = pd.DataFrame(
    [
    ['litro', 'litro', 1],
    ['gramos', 'kilogramo', 1000],
    ['kilogramos', 'kilogramo', 1],
    ['litros', 'litro', 1],
    ['kilogramo', 'kilogramo', 1],
    ['centímetros cúbicos', 'litro', 1000]
    ],
    columns=['u_entrada', 'u_salida', 'factor_conversion'])

display(factores_conversión)

,u_entrada,u_salida,factor_conversion
0,litro,litro,1
1,gramos,kilogramo,1000
2,kilogramos,kilogramo,1
3,litros,litro,1
4,kilogramo,kilogramo,1
5,centímetros cúbicos,litro,1000


In [12]:
# Merge de factores de conversión con el dataframe
df_insumos_agricolas = df_insumos_agricolas.merge(
    factores_conversión,
    left_on='unidad',
    right_on='u_entrada',
    how='left',
    indicator=True
)

display(df_insumos_agricolas['_merge'].value_counts())

_merge
both          582365
left_only          0
right_only         0
Name: count, dtype: int64

In [13]:
# calcular precio unitario en las mismas unidades
df_insumos_agricolas['precio_unitario'] = (df_insumos_agricolas['Precio promedio ']/df_insumos_agricolas['cantidad']
                                           *df_insumos_agricolas['factor_conversion']
                                           )

# Verificación manual
df_insumos_agricolas[[
    'tipo_insumo',
    'Nombre del producto',
    'Precio promedio ',
    'Presentación del producto',
    'precio_unitario',
    'u_salida',
]]
# Verificar manualmente que las unidades hayan quedado bien convertidas
# Por ejemplo, 
# Safermix cuesta 42,000 COP / 500 gramos, es decir que el precio unitario (en kg) es 82,000 COP/kg
# Voliam Flexi cuesta 119,375 COP / 250 c.c. , es decir que el precio unitario (en litros) es 477,500 COP/litro

,tipo_insumo,Nombre del producto,Precio promedio,Presentación del producto,precio_unitario,u_salida
0,Bioinsumos,Alisin,"40,666.67",1 litro,"40,666.67",litro
1,Bioinsumos,Alisin,"43,050.00",1 litro,"43,050.00",litro
2,Bioinsumos,Sáfermix Wp,"42,800.00",500 gramos,"85,600.00",kilogramo
3,Bioinsumos,Sáfermix Wp,"41,000.00",500 gramos,"82,000.00",kilogramo
4,Bioinsumos,Sáfermix Wp,"42,000.00",500 gramos,"84,000.00",kilogramo
...,...,...,...,...,...,...
582360,"Insecticidas, acaricidas y nematicidas",Voliam Flexi,"119,375.00",250 centímetros cúbicos,"477,500.00",litro
582361,"Insecticidas, acaricidas y nematicidas",Voliam Flexi,"122,850.00",250 centímetros cúbicos,"491,400.00",litro
582362,"Insecticidas, acaricidas y nematicidas",Voliam Flexi,"2,288,400.00",5 litros,"457,680.00",litro
582363,"Insecticidas, acaricidas y nematicidas",Voliam Flexi,"2,050,876.50",5 litros,"410,175.30",litro


In [14]:
# Verificar para uno de los insumos con más presentaciones de la tabla de resumen de arriba: "Panzer 480 Sl"
mascara = df_insumos_agricolas['Nombre del producto']=="Panzer 480 Sl"

df_insumos_agricolas[['Nombre del producto', 'Precio promedio ', 'Presentación del producto', 'precio_unitario', 'u_salida']][mascara]

,Nombre del producto,Precio promedio,Presentación del producto,precio_unitario,u_salida
323562,Panzer 480 Sl,"16,188.75",1 litro,"16,188.75",litro
323563,Panzer 480 Sl,"14,684.00",1 litro,"14,684.00",litro
323564,Panzer 480 Sl,"15,700.00",1 litro,"15,700.00",litro
323565,Panzer 480 Sl,"14,103.33",1 litro,"14,103.33",litro
323566,Panzer 480 Sl,"14,792.00",1 litro,"14,792.00",litro
...,...,...,...,...,...
452427,Panzer 480 Sl,"74,666.67",4 litros,"18,666.67",litro
452428,Panzer 480 Sl,"67,666.67",4 litros,"16,916.67",litro
452429,Panzer 480 Sl,"75,333.33",4 litros,"18,833.33",litro
452430,Panzer 480 Sl,"72,500.00",4 litros,"18,125.00",litro


In [15]:
# Como ya se calcularon los precios unitarios, hago el promedio de las observaciones por municipio por mes por producto
# CONSIDERACIÓN
# Cuando se compra en presentaciones grandes, el precio unitario tiende a ser menor
# Si saco el promedio de los datos observados en el municipio, los municipios donde se tienda a vender las presentaciones grandes sobre las pequeñas, van a tener un menor precio unitario
# Si el muestreo del DANE refleja las presentaciones que se encuentran fácilmente en el municipio en lugar de que simplemente elijan arbitrariamente una presentación por cada municipio,
# esto reflejaría municipios donde se tiende a comprar por presentaciones grandes y en consecuencia tienen menores costos unitarios.
# Si el muestreo en el municipio es consistente a lo largo del tiempo y calculo un índice de precios, tampoco debería tener problema.

# Colapsar series mensuales a series anuales
# Para efectos de la propuesta, colapso los datos a promedios simples
panel_insumos = df_insumos_agricolas.groupby(['Código municipio', 'Año ', 'tipo_insumo', 'Nombre del producto', 'u_salida'])['precio_unitario'].mean().reset_index()
display(panel_insumos.head(3))

,Código municipio,Año,tipo_insumo,Nombre del producto,u_salida,precio_unitario
0,05001,2021,"Coadyuvantes, molusquicidas, reguladores fisio...",Agrotin Sl,litro,"20,642.47"
1,05001,2021,"Coadyuvantes, molusquicidas, reguladores fisio...",Baboxa Sb,kilogramo,"28,175.00"
2,05001,2021,"Coadyuvantes, molusquicidas, reguladores fisio...",Carrier,litro,"22,734.28"


In [16]:
# Para organizar los datos en el PANEL, necesito que todo quede con la misma estructura.
# Las columnas del Panel van a ser

# Codimo_mpio --- Año   --- Variable                        --- Valor  --- Auxiliar         --- Tipo_variable

# NNNN        --- 2015  --- 'Aguacate_Hass_Producción (t)'  --- 1,000  --- Producción (t) --- Resultado
# NNNN        --- 2015  --- 'No_Aguacate_Producción (t)'    --- 1,000  --- Producción (t) --- Control - Otros cultivos
# NNNN        --- 2015  --- 'Area_titulo_minero_m2'         --- 20,000 --- m2             --- Instrumento
# NNNN        --- 2015  --- 'precio_15-15-15_COP/kg'        --- 20,000 --- Fertilizantes  --- Control - Insumos agrícolas

# Organizo los nombres anteriores del DF en los nombres que necesito para armar el panel
panel_insumos['variable'] = 'precio_' + panel_insumos['Nombre del producto'] + '_COP/' + panel_insumos['u_salida']
panel_insumos['auxiliar'] = panel_insumos['tipo_insumo']
panel_insumos['tipo_variable'] = 'Control - Insumos agrícolas'
panel_insumos = panel_insumos.rename(columns= {'precio_unitario': 'valor'})

In [17]:
# Mostrar panel
display(panel_insumos.drop(columns={'Nombre del producto', 'tipo_insumo', 'u_salida'}).head())

,Código municipio,Año,valor,variable,auxiliar,tipo_variable
0,05001,2021,"20,642.47",precio_Agrotin Sl_COP/litro,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
1,05001,2021,"28,175.00",precio_Baboxa Sb_COP/kilogramo,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
2,05001,2021,"22,734.28",precio_Carrier_COP/litro,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
3,05001,2021,"890,234.38",precio_Marax_COP/kilogramo,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
4,05001,2021,"13,516.67",precio_Matababosa 7%_COP/kilogramo,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas


### Precio jornales

In [18]:
# Definir estructura de los datos que voy a cargar
estructura_datos = pd.DataFrame([
    ['Arrendamiento', '3.1', 'Tipo de arriendo ', False],
    ['Distrito de riego', '3.2', 'Distrito de riego', 'Tipo de pago '],
    ['Elementos agropecuarios', '3.3', 'Nombre del producto', 'Presentación del producto'],
    ['Empaques', '3.4', 'Nombre del producto', 'Presentación del producto'],
    ['Especies productivas', '3.6', 'Especie productiva ', 'Presentación'],
    ['Jornales', '3.7', 'Tipo de jornal', False],
    ['Material propagación', '3.8', 'Nombre del producto', 'Presentación del producto'],
    ['Servicios Agricolas', '3.9', 'Nombre del servicio', 'Tipo de servicio']
], columns=['tipo_insumo', 'hoja_excel', 'insumo', 'Presentacion'])
estructura_datos

,tipo_insumo,hoja_excel,insumo,Presentacion
0,Arrendamiento,3.1,Tipo de arriendo,False
1,Distrito de riego,3.2,Distrito de riego,Tipo de pago
2,Elementos agropecuarios,3.3,Nombre del producto,Presentación del producto
3,Empaques,3.4,Nombre del producto,Presentación del producto
4,Especies productivas,3.6,Especie productiva,Presentación
5,Jornales,3.7,Tipo de jornal,False
6,Material propagación,3.8,Nombre del producto,Presentación del producto
7,Servicios Agricolas,3.9,Nombre del servicio,Tipo de servicio


In [19]:
# Crear DF vacío para almacenar los datos
df_insumos_produccion = pd.DataFrame()


for _,row in tqdm(estructura_datos.iterrows()):
    
    # cargar cada una de las hojas del archivo
    df_insumos = pd.read_excel(
        ruta_insumos_agropecuarios,
        sheet_name=row['hoja_excel'],
        skiprows=8,
        dtype={'Código municipio':str}
    )
    
    # Eliminar las las notas del DANE (filas del final). Son las que tienen NAN en la 
    # columna 'Código municipio'
    df_insumos = df_insumos.dropna(subset='Código municipio')
    
    # Agregar identificador del tipo de insumo
    df_insumos['tipo_insumo'] = row['tipo_insumo']
    
    # Cambiar nombre de las columnas como se definió en la celda anterior
    nombre_columna = row['insumo']
    diccionario_renombrar = {nombre_columna:'insumo'}
    df_insumos = df_insumos.rename(columns=diccionario_renombrar)
    
    # Cambiar nombre de las columnas como se definió en la celda anterior
    # No todas las hojas tienen una columna para estos datos
    if row['Presentacion'] != False:
        nombre_columna = row['Presentacion']
        diccionario_renombrar = {nombre_columna:'Presentacion'}
        df_insumos = df_insumos.rename(columns=diccionario_renombrar)
    
    # Uniformar los nombres de las columnas
    df_insumos.columns = df_insumos.columns.str.strip().str.replace(' ', '_')
    
    # Unir las observaciones en un solo df
    df_insumos_produccion = pd.concat(
        [df_insumos_produccion, df_insumos],
        axis=0,
        ignore_index=True
    )
    
# Eliminar columnas sin información
df_insumos_produccion = df_insumos_produccion.drop(columns=['Unnamed:_8', 'Unnamed:_9'])

8it [00:13,  1.75s/it]


In [20]:
# Explorar datos cargados
df_insumos_produccion.head()

,Año,Mes,Código_departamento,Nombre_departamento,Código_municipio,Nombre_municipio,insumo,Precio_promedio,tipo_insumo,Presentacion
0,2021,Febrero,19.00,Cauca,19001,Popayán,"Cabeza de ganado mensual para pastoreo, terren...","20,666.67",Arrendamiento,NaN
1,2021,Febrero,19.00,Cauca,19548,Piendamó - Tunía,"Cabeza de ganado mensual para pastoreo, terren...","20,000.00",Arrendamiento,NaN
2,2021,Febrero,23.00,Córdoba,23417,Lorica,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
3,2021,Febrero,68.00,Santander,68406,Lebrija,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
4,2021,Febrero,68.00,Santander,68547,Piedecuesta,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN


In [21]:
df_insumos_produccion[df_insumos_produccion['tipo_insumo']=='Arrendamiento']

,Año,Mes,Código_departamento,Nombre_departamento,Código_municipio,Nombre_municipio,insumo,Precio_promedio,tipo_insumo,Presentacion
0,2021,Febrero,19.00,Cauca,19001,Popayán,"Cabeza de ganado mensual para pastoreo, terren...","20,666.67",Arrendamiento,NaN
1,2021,Febrero,19.00,Cauca,19548,Piendamó - Tunía,"Cabeza de ganado mensual para pastoreo, terren...","20,000.00",Arrendamiento,NaN
2,2021,Febrero,23.00,Córdoba,23417,Lorica,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
3,2021,Febrero,68.00,Santander,68406,Lebrija,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
4,2021,Febrero,68.00,Santander,68547,Piedecuesta,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
...,...,...,...,...,...,...,...,...,...,...
4997,2026,Febrero,63.00,Quindío,63470,Montenegro,"Hectárea semestral, terreno quebrado sin agua","656,666.67",Arrendamiento,NaN
4998,2026,Febrero,15.00,Boyacá,15087,Belén,Terreno para cultivo carga de papa (100 kilogr...,"130,000.00",Arrendamiento,NaN
4999,2026,Febrero,15.00,Boyacá,15861,Ventaquemada,Terreno para cultivo carga de papa (100 kilogr...,"100,000.00",Arrendamiento,NaN
5000,2026,Febrero,25.00,Cundinamarca,25183,Chocontá,Terreno para cultivo carga de papa (100 kilogr...,"120,000.00",Arrendamiento,NaN


In [22]:
df_insumos_produccion.groupby('tipo_insumo')['Presentacion'].nunique().reset_index().sort_values('Presentacion', ascending=False)

,tipo_insumo,Presentacion
1,Distrito de riego,55
2,Elementos agropecuarios,40
6,Material propagación,27
7,Servicios Agricolas,7
3,Empaques,4
4,Especies productivas,4
0,Arrendamiento,0
5,Jornales,0


In [23]:
df_insumos_produccion.groupby('insumo')['Presentacion'].nunique().reset_index().sort_values('Presentacion', ascending=False)

,insumo,Presentacion
543,Usocoello,12
75,Asorrecio,11
546,Usosaldaña,9
536,Toruro,4
67,Asodistricharte,4
...,...,...
354,"Hectárea semestral para maíz, terreno plano co...",0
355,"Hectárea semestral para maíz, terreno plano o ...",0
356,"Hectárea semestral para maíz, terreno plano si...",0
357,"Hectárea semestral para papa, terreno plano co...",0


In [24]:
# Dada la diversidad de insumos de producción agrícola, para la propuesta me voy a limitar a Jornales, Arrendamiento, especies productivas
mascara = df_insumos_produccion['tipo_insumo'].isin(['Arrendamiento', 'Jornales', 'Servicios Agricolas', 'Especies productivas'])
seleccion_insumos_produccion = df_insumos_produccion[mascara]
display(seleccion_insumos_produccion.head())

,Año,Mes,Código_departamento,Nombre_departamento,Código_municipio,Nombre_municipio,insumo,Precio_promedio,tipo_insumo,Presentacion
0,2021,Febrero,19.00,Cauca,19001,Popayán,"Cabeza de ganado mensual para pastoreo, terren...","20,666.67",Arrendamiento,NaN
1,2021,Febrero,19.00,Cauca,19548,Piendamó - Tunía,"Cabeza de ganado mensual para pastoreo, terren...","20,000.00",Arrendamiento,NaN
2,2021,Febrero,23.00,Córdoba,23417,Lorica,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
3,2021,Febrero,68.00,Santander,68406,Lebrija,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN
4,2021,Febrero,68.00,Santander,68547,Piedecuesta,"Cabeza de ganado mensual para pastoreo, terren...","30,000.00",Arrendamiento,NaN


In [25]:
# Para organizar los datos en el PANEL, necesito que todo quede con la misma estructura.
# Las columnas del Panel van a ser

# Codimo_mpio --- Año   --- Variable                        --- Valor  --- Auxiliar         --- Tipo_variable

# NNNN        --- 2015  --- 'Aguacate_Hass_Producción (t)'  --- 1,000  --- Producción (t) --- Resultado
# NNNN        --- 2015  --- 'No_Aguacate_Producción (t)'    --- 1,000  --- Producción (t) --- Control - Otros cultivos
# NNNN        --- 2015  --- 'Area_titulo_minero_m2'         --- 20,000 --- m2             --- Instrumento
# NNNN        --- 2015  --- 'precio_15-15-15_COP/kg'        --- 20,000 --- Fertilizantes  --- Control - Insumos agrícolas
# NNNN        --- 2015  --- 'precio_jornal_COP'             --- 40,000 --- jornales       --- Control - insumos agropecuarios

# Agregar por año
panel_seleccion_insumos_produccion = seleccion_insumos_produccion.groupby(['Año', 'Código_municipio', 'insumo', 'tipo_insumo', 'Presentacion'])['Precio_promedio'].mean().reset_index()
panel_seleccion_insumos_produccion['variable'] = panel_seleccion_insumos_produccion['insumo'] + '_' + panel_seleccion_insumos_produccion['Presentacion']
panel_seleccion_insumos_produccion['valor'] = panel_seleccion_insumos_produccion['Precio_promedio']
panel_seleccion_insumos_produccion['auxiliar'] = panel_seleccion_insumos_produccion['tipo_insumo']
panel_seleccion_insumos_produccion['tipo_variable'] = 'Control - insumos agropecuarios'
panel_seleccion_insumos_produccion.drop(columns=['insumo', 'tipo_insumo', 'Presentacion', 'Precio_promedio']).head()

,Año,Código_municipio,variable,valor,auxiliar,tipo_variable
0,2021,05001,Bovino Cebú macho 13-18 meses_kilogramo,"6,786.25",Especies productivas,Control - insumos agropecuarios
1,2021,05001,Bovino Cebú macho 19-24 meses_kilogramo,"5,750.00",Especies productivas,Control - insumos agropecuarios
2,2021,05001,Cerdo Pietran hembra-macho 50-60 kilogramos_ki...,"7,250.00",Especies productivas,Control - insumos agropecuarios
3,2021,05001,Pollo/Polla engorde Ross hembra-macho 1 día_un...,"2,158.33",Especies productivas,Control - insumos agropecuarios
4,2021,05034,Pollo/Polla engorde Ross hembra-macho 1 día_un...,"2,088.89",Especies productivas,Control - insumos agropecuarios


## Producción asociada a regalías

In [26]:
# Cargar datos
produccion_regalias = pd.read_excel('../../Data/10_UPME_ANM_Producción_regalías/PyRMC.xlsm',
                                   sheet_name='PyR',
                                   skiprows=1,
                                   dtype={'Codigo Municipio':str})

display(produccion_regalias.head())

,Año,Mes,Mineral,UnidadMedida,Regalias ($),Produccion,Codigo Municipio,Municipio,Codigo Departamento,Departamento
0,2026,1,ORO,GRAMOS,"1,013,843,150.81","634,108.02",05736,SEGOVIA,5,ANTIOQUIA
1,2025,12,ORO,GRAMOS,"969,828,842.90","690,459.19",05736,SEGOVIA,5,ANTIOQUIA
2,2025,11,ORO,GRAMOS,"746,090,262.90","612,155.49",05736,SEGOVIA,5,ANTIOQUIA
3,2025,10,ORO,GRAMOS,"705,220,681.15","567,809.15",05736,SEGOVIA,5,ANTIOQUIA
4,2025,9,ORO,GRAMOS,"885,010,981.44","788,920.45",05736,SEGOVIA,5,ANTIOQUIA


In [27]:
# Para organizar los datos en el PANEL, necesito que todo quede con la misma estructura.
# Las columnas del Panel van a ser

# Codimo_mpio --- Año   --- Variable                        --- Valor  --- Auxiliar         --- Tipo_variable

# NNNN        --- 2015  --- 'Aguacate_Hass_Producción (t)'  --- 1,000  --- Producción (t) --- Resultado
# NNNN        --- 2015  --- 'No_Aguacate_Producción (t)'    --- 1,000  --- Producción (t) --- Control - Otros cultivos
# NNNN        --- 2015  --- 'Area_titulo_minero_m2'         --- 20,000 --- m2             --- Instrumento
# NNNN        --- 2015  --- 'precio_15-15-15_COP/kg'        --- 20,000 --- Fertilizantes  --- Control - Insumos agrícolas
# NNNN        --- 2015  --- 'precio_jornal_COP'             --- 40,000 --- jornales       --- Control - insumos agropecuarios
# NNNN        --- 2015  --- 'P_regalias_oro_gramos'         --- 5,000  --- Oro            --- Tratamiento - producción regalías

# Agregar por año
panel_produccion_regalias = produccion_regalias.groupby([
    'Año', 'Mineral', 'UnidadMedida', 'Codigo Municipio'
])['Produccion'].sum().reset_index()

# Organizar datos en el formato requerido
panel_produccion_regalias['valor'] = panel_produccion_regalias['Produccion']
panel_produccion_regalias['variable'] = 'Produccion_' + panel_produccion_regalias['Mineral'] + '_' + panel_produccion_regalias['UnidadMedida']
panel_produccion_regalias['auxiliar'] = panel_produccion_regalias['Mineral']
panel_produccion_regalias['tipo_variable'] = 'Tratamiento - producción regalías'
panel_produccion_regalias.drop(columns=['Produccion', 'Mineral', 'UnidadMedida']).head()


,Año,Codigo Municipio,valor,variable,auxiliar,tipo_variable
0,2012,05030,0.00,Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías
1,2012,05360,"4,517.00",Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías
2,2012,11001,0.00,Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías
3,2012,15759,0.00,Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías
4,2012,15763,"13,525.00",Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías


# Armar Panel


## Variables de resultado
- UPRA-EVA (Area y produccion de Café o aguacate) `panel_cultivos`

In [48]:
PANEL = None

In [50]:
# Empezar el panel con las variables de resultado
PANEL = panel_cultivos.copy()

# Limpiar variables del panel
PANEL = PANEL[['Código Dane municipio', 'Año', 'variable', 'valor', 'auxiliar', 'tipo_variable']]

display(PANEL.info()) # Revisar estructura de los datos
print(f"\n{'*'*20}")
display(PANEL.head(2))
print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238542 entries, 0 to 238541
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  238542 non-null  object 
 1   Año                    238542 non-null  int32  
 2   variable               238542 non-null  object 
 3   valor                  238542 non-null  float64
 4   auxiliar               238542 non-null  object 
 5   tipo_variable          238542 non-null  object 
dtypes: float64(1), int32(1), object(4)
memory usage: 10.0+ MB


None


********************


,Código Dane municipio,Año,variable,valor,auxiliar,tipo_variable
0,05001,2019,Aguacate demás variedades_Área sembrada (ha),24.00,Área sembrada (ha),Resultado
1,05001,2020,Aguacate demás variedades_Área sembrada (ha),8.52,Área sembrada (ha),Resultado



**************************************************
Resumen del PANEL
**************************************************


variable   Año       Código Dane municipio   valor
                          nunique   min   max               nunique   count
tipo_variable                                                              
Control - Otros cultivos       48  2019  2024                  1102  212868
Resultado                      12  2019  2024                   744   25674

## Tratamiento
- Producción asociada a regalias `panel_produccion_regalias`

In [51]:
# Arreglar nombre de las columnas
display(panel_produccion_regalias.head(2)) # Mostrar datos para elegir columnas columnas y corregir nombres
panel_agregar = panel_produccion_regalias[['Codigo Municipio', 'Año', 'variable', 'valor', 'auxiliar', 'tipo_variable']].rename(columns={'Codigo Municipio':'Código Dane municipio'})

# Mostrar datos
print(f"\n{'*'*20}")
display(panel_agregar.head(2))
print(f"\n{'*'*20}")
display(panel_agregar.info()) # Revisar estructura de los datos


# Agregar en estructura PANEL
print(f"\n{'*'*50} ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} DESPUÉS de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})

,Año,Mineral,UnidadMedida,Codigo Municipio,Produccion,valor,variable,auxiliar,tipo_variable
0,2012,ARCILLAS,TONELADAS,05030,0.00,0.00,Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías
1,2012,ARCILLAS,TONELADAS,05360,"4,517.00","4,517.00",Produccion_ARCILLAS_TONELADAS,ARCILLAS,Tratamiento - producción regalías



********************


,Código Dane municipio,Año,variable,valor,auxiliar,tipo_variable
0,05030,2012,Produccion_ARCILLAS_TONELADAS,0.00,ARCILLAS,Tratamiento - producción regalías
1,05360,2012,Produccion_ARCILLAS_TONELADAS,"4,517.00",ARCILLAS,Tratamiento - producción regalías



********************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22638 entries, 0 to 22637
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  22638 non-null  object 
 1   Año                    22638 non-null  int64  
 2   variable               22638 non-null  object 
 3   valor                  22638 non-null  float64
 4   auxiliar               22638 non-null  object 
 5   tipo_variable          22638 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.0+ MB


None

...

None


************************************************** DESPUÉS de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 261180 entries, 0 to 22637
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  261180 non-null  object 
 1   Año                    261180 non-null  int64  
 2   variable               261180 non-null  object 
 3   valor                  261180 non-null  float64
 4   auxiliar               261180 non-null  object 
 5   tipo_variable          261180 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 13.9+ MB


None


**************************************************
Resumen del PANEL
**************************************************


variable   Año       Código Dane municipio  \
                                   nunique   min   max               nunique   
tipo_variable                                                                  
Control - Otros cultivos                48  2019  2024                  1102   
Resultado                               12  2019  2024                   744   
Tratamiento - producción regalías       76  2012  2026                   881   

                                    valor  
                                    count  
tipo_variable                              
Control - Otros cultivos           212868  
Resultado                           25674  
Tratamiento - producción regalías   22638

## Instrumentos
- Area titulo minero (PENDIENTE)
- area potencial minero (PENDIENTE)
- precio del oro (PENDIENTE)
- mineros de subsistencia (PENDIENTE)
- Instrumento de interacción Potencial X Precio oro (PENDIENTE)

### Área título minero

In [52]:
# Cargar datos
panel_area_titulo_minero = pd.read_parquet('../2_estimar_area_titulo_minero/outputs/PANEL_AREA_TITULO_MINERO_STOCK_FLUJO.parquet')

# Mostrar estructura
display(panel_area_titulo_minero.info())
print(f"\n{'*'*20}")
display(panel_area_titulo_minero.head(2))

# Simplificar y Arreglar nombre de las columnas
panel_area_titulo_minero = panel_area_titulo_minero[['mpio_cdpmp', 'anno', 'area_solicitada_stock_m2', 'area_titulada_stock_m2', 'tipo_mineria']].rename(
    columns={'mpio_cdpmp':'Código Dane municipio',
            'anno':'Año'})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8184 entries, 0 to 8183
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   mpio_cdpmp                8184 non-null   object 
 1   anno                      8184 non-null   int64  
 2   tipo_mineria              8184 non-null   object 
 3   area_solicitada_m2        8184 non-null   float64
 4   area_expedida_m2          8184 non-null   float64
 5   area_expirada_m2          8184 non-null   float64
 6   area_titulada_m2          8184 non-null   float64
 7   area_solicitada_stock_m2  8184 non-null   float64
 8   area_titulada_stock_m2    8184 non-null   float64
dtypes: float64(6), int64(1), object(2)
memory usage: 575.6+ KB


None


********************


,mpio_cdpmp,anno,tipo_mineria,area_solicitada_m2,area_expedida_m2,area_expirada_m2,area_titulada_m2,area_solicitada_stock_m2,area_titulada_stock_m2
0,05004,1989,Grande,0.00,0.00,0.00,0.00,0.00,0.00
1,05004,1989,Otras,0.00,0.00,0.00,0.00,0.00,0.00


Dar estructura a los datos para acoplarse al PANEL

|Código Dane municipio|Año|variable|valor|auxiliar|tipo_variable|
|-|-|-|-|-|-|
05003|2015|Aguacate_Hass_Producción (t)|20|Producción (t)|Resultado
05003|2015|No_Aguacate_Producción (t)|100|Producción (t)|Control - Otros cultivos
05003|2015|precio_15-15-15_COP/kg|20,000|Fertilizantes|Control - Insumos agrícolas
05003|2015|precio_jornal_COP|50,000|jornales|Control - insumos agropecuarios
05003|2015|P_regalias_oro_gramos|5,000|Oro|Tratamiento - producción regalías
05003|2015|**area_titulada_stock_m2**|30,000|**Grande**|Instrumento - area titulada




In [53]:
# Dar estructura a los datos para acoplarse al PANEL
panel_agregar = panel_area_titulo_minero.melt(id_vars=['Código Dane municipio', 'Año', 'tipo_mineria'],
                             value_vars=['area_solicitada_stock_m2', 'area_titulada_stock_m2'],
                             value_name='valor')

# organizar columnas
panel_agregar = panel_agregar.rename(columns={'tipo_mineria':'auxiliar'})
panel_agregar['tipo_variable'] = 'Instrumento - area titulo minero'


# Mostrar datos
print(f"\n{'*'*50} Variable a agregar")
display(panel_agregar.head(2))
print(f"\n{'*'*50}")
display(panel_agregar.info()) # Revisar estructura de los datos


# Agregar en estructura PANEL
print(f"\n{'*'*50} PANEL ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# # Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} PANEL DESPUÉS de nuevas variables")
# display(PANEL.info()) # Revisar estructura de los datos

print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})


************************************************** Variable a agregar


,Código Dane municipio,Año,auxiliar,variable,valor,tipo_variable
0,05004,1989,Grande,area_solicitada_stock_m2,0.00,Instrumento - area titulo minero
1,05004,1989,Otras,area_solicitada_stock_m2,0.00,Instrumento - area titulo minero



**************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16368 entries, 0 to 16367
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  16368 non-null  object 
 1   Año                    16368 non-null  int64  
 2   auxiliar               16368 non-null  object 
 3   variable               16368 non-null  object 
 4   valor                  16368 non-null  float64
 5   tipo_variable          16368 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 767.4+ KB


None


************************************************** PANEL ANTES de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 261180 entries, 0 to 22637
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  261180 non-null  object 
 1   Año                    261180 non-null  int64  
 2   variable               261180 non-null  object 
 3   valor                  261180 non-null  float64
 4   auxiliar               261180 non-null  object 
 5   tipo_variable          261180 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 13.9+ MB


None


************************************************** PANEL DESPUÉS de nuevas variables

**************************************************
Resumen del PANEL
**************************************************


variable   Año       Código Dane municipio  \
                                   nunique   min   max               nunique   
tipo_variable                                                                  
Control - Otros cultivos                48  2019  2024                  1102   
Instrumento - area titulo minero         2  1989  2021                   124   
Resultado                               12  2019  2024                   744   
Tratamiento - producción regalías       76  2012  2026                   881   

                                    valor  
                                    count  
tipo_variable                              
Control - Otros cultivos           212868  
Instrumento - area titulo minero    16368  
Resultado                           25674  
Tratamiento - producción regalías   22638

### Potencial minero

In [54]:
# Cargar y explorar datos
df_potencial_minero = pd.read_parquet('../1_estimar_porcentaje_area_potencial_minero/porcentaje_area_municipal_potencial_minero.parquet')

# Mostrar estructura
display(df_potencial_minero.info())
print(f"\n{'*'*20}")
display(df_potencial_minero.head(2))

# Simplificar y Arreglar nombre de las columnas
df_potencial_minero = df_potencial_minero[['mpio_cdpmp', 'pct_municipio_cubierto']].rename(
    columns={'mpio_cdpmp':'Código Dane municipio'})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dpto_ccdgo              1122 non-null   object 
 1   mpio_ccdgo              1122 non-null   object 
 2   mpio_cdpmp              1122 non-null   object 
 3   dpto_cnmbr              1122 non-null   object 
 4   mpio_cnmbr              1122 non-null   object 
 5   mpio_crslc              1122 non-null   object 
 6   mpio_tipo               1122 non-null   object 
 7   mpio_narea              1122 non-null   float64
 8   mpio_nano               1122 non-null   int32  
 9   shape_Leng              1122 non-null   float64
 10  shape_Area              1122 non-null   float64
 11  geometry                1122 non-null   object 
 12  pct_municipio_cubierto  1122 non-null   float64
 13  pct_plot                445 non-null    float64
dtypes: float64(5), int32(1), object(8)
memor

None


********************


,dpto_ccdgo,mpio_ccdgo,mpio_cdpmp,dpto_cnmbr,mpio_cnmbr,mpio_crslc,mpio_tipo,mpio_narea,mpio_nano,shape_Leng,shape_Area,geometry,pct_municipio_cubierto,pct_plot
0,05,001,05001,ANTIOQUIA,MEDELLÍN,1965,MUNICIPIO,374.74,2025,1.04,0.03,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\xed \x0...,0.00,NaN
1,05,002,05002,ANTIOQUIA,ABEJORRAL,1814,MUNICIPIO,506.96,2024,1.16,0.04,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00*\x0e\x0...,89.01,89.01


**Dar estructura a los datos para acoplarse al PANEL**

|Código Dane municipio|Año|variable|valor|auxiliar|tipo_variable|
|-|-|-|-|-|-|
05003|2015|Aguacate_Hass_Producción (t)|20|Producción (t)|Resultado
05003|2015|No_Aguacate_Producción (t)|100|Producción (t)|Control - Otros cultivos
05003|2015|precio_15-15-15_COP/kg|20,000|Fertilizantes|Control - Insumos agrícolas
05003|2015|precio_jornal_COP|50,000|jornales|Control - insumos agropecuarios
05003|2015|P_regalias_oro_gramos|5,000|Oro|Tratamiento - producción regalías
05003|2015|area_titulada_stock_m2|30,000|Grande|Instrumento - area titulada
05003|2015|**pct_municipio_potencial_oro**|98|Oro|Instrumento - area potencial oro

In [55]:
# Dar estructura a los datos para acoplarse al PANEL
df_potencial_minero['Año'] = 2026 # Todos los años tienen el mismo valor. Acá elijo cualqueira
df_potencial_minero['auxiliar'] = 'Oro'
df_potencial_minero['variable'] = 'pct_municipio_potencial_oro'
df_potencial_minero['tipo_variable'] = 'Instrumento - potencial geológico oro'

panel_agregar = df_potencial_minero.rename(columns={'pct_municipio_cubierto':'valor'})

# Mostrar datos
print(f"\n{'*'*50} Variable a agregar")
display(panel_agregar.head(2))
print(f"\n{'*'*50}")
display(panel_agregar.info()) # Revisar estructura de los datos


# Agregar en estructura PANEL
print(f"\n{'*'*50} PANEL ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# # Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} PANEL DESPUÉS de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})


************************************************** Variable a agregar


,Código Dane municipio,valor,Año,auxiliar,variable,tipo_variable
0,05001,0.00,2026,Oro,pct_municipio_potencial_oro,Instrumento - potencial geológico oro
1,05002,89.01,2026,Oro,pct_municipio_potencial_oro,Instrumento - potencial geológico oro



**************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  1122 non-null   object 
 1   valor                  1122 non-null   float64
 2   Año                    1122 non-null   int64  
 3   auxiliar               1122 non-null   object 
 4   variable               1122 non-null   object 
 5   tipo_variable          1122 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 52.7+ KB


None


************************************************** PANEL ANTES de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 277548 entries, 0 to 16367
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  277548 non-null  object 
 1   Año                    277548 non-null  int64  
 2   variable               277548 non-null  object 
 3   valor                  277548 non-null  float64
 4   auxiliar               277548 non-null  object 
 5   tipo_variable          277548 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 14.8+ MB


None


************************************************** PANEL DESPUÉS de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 278670 entries, 0 to 1121
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  278670 non-null  object 
 1   Año                    278670 non-null  int64  
 2   variable               278670 non-null  object 
 3   valor                  278670 non-null  float64
 4   auxiliar               278670 non-null  object 
 5   tipo_variable          278670 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 14.9+ MB


None


**************************************************
Resumen del PANEL
**************************************************


variable   Año        \
                                       nunique   min   max   
tipo_variable                                                
Control - Otros cultivos                    48  2019  2024   
Instrumento - area titulo minero             2  1989  2021   
Instrumento - potencial geológico oro        1  2026  2026   
Resultado                                   12  2019  2024   
Tratamiento - producción regalías           76  2012  2026   

                                      Código Dane municipio   valor  
                                                    nunique   count  
tipo_variable                                                        
Control - Otros cultivos                               1102  212868  
Instrumento - area titulo minero                        124   16368  
Instrumento - potencial geológico oro                  1122    1122  
Resultado                                               744   25674  
Tratamiento - producción regalías                       881   22638

### Minería de subsistencia

**Dar estructura a los datos para acoplarse al PANEL**

|Código Dane municipio|Año|variable|valor|auxiliar|tipo_variable|
|-|-|-|-|-|-|
05003|2015|Aguacate_Hass_Producción (t)|20|Producción (t)|Resultado
05003|2015|No_Aguacate_Producción (t)|100|Producción (t)|Control - Otros cultivos
05003|2015|precio_15-15-15_COP/kg|20,000|Fertilizantes|Control - Insumos agrícolas
05003|2015|precio_jornal_COP|50,000|jornales|Control - insumos agropecuarios
05003|2015|P_regalias_oro_gramos|5,000|Oro|Tratamiento - producción regalías
05003|2015|area_titulada_stock_m2|30,000|Grande|Instrumento - area titulada
05003|2015|pct_municipio_potencial_oro|98|Oro|Instrumento - area potencial oro
05003|2015|LOG_subsistencia_METALES_PRECIOSOS_x10khab|533|METALES_PRECIOSOS|Instrumento - mineros subsistencia

In [56]:
# Cargar y explorar datos
df_mineros_subsitencia = pd.read_parquet('../4_explorar_presencia_mineros_subsistencia/outputs/D1_mineros_subsistencia_x10000habitantes_NivObs_Municipio.parquet')

# Mostrar estructura
display(df_mineros_subsitencia.info())
print(f"\n{'*'*20}")
display(df_mineros_subsitencia.head(2))

# Simplificar y Arreglar nombre de las columnas
df_mineros_subsitencia = df_mineros_subsitencia[['MPIO',
                                                 'LOG_subsistencia_ARCILLA_x10khab',
                                                 'LOG_subsistencia_ARENAS_DE_RIO_x10khab',
                                                 'LOG_subsistencia_ESMERALDAS_x10khab',
                                                 'LOG_subsistencia_GRAVAS_DE_RIO_x10khab',
                                                 'LOG_subsistencia_METALES_PRECIOSOS_x10khab',
                                                 'LOG_subsistencia_MORRALLA_x10khab',
                                                 'LOG_subsistencia_PIEDRAS_PRECIOSAS_x10khab',
                                                 'LOG_subsistencia_PIEDRAS_SEMIPRECIOSAS_x10khab']]
df_mineros_subsitencia = df_mineros_subsitencia.rename(columns={'MPIO':'Código Dane municipio'})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1123 entries, 0 to 1122
Data columns (total 26 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   subsistencia_ARCILLA                            1123 non-null   float64
 1   subsistencia_ARENAS_DE_RIO                      1123 non-null   float64
 2   subsistencia_ESMERALDAS                         1123 non-null   float64
 3   subsistencia_GRAVAS_DE_RIO                      1123 non-null   float64
 4   subsistencia_METALES_PRECIOSOS                  1123 non-null   float64
 5   subsistencia_MORRALLA                           1123 non-null   float64
 6   subsistencia_PIEDRAS_PRECIOSAS                  1123 non-null   float64
 7   subsistencia_PIEDRAS_SEMIPRECIOSAS              1123 non-null   float64
 8   MPIO                                            1123 non-null   object 
 9   Poblacion                                

None


********************


,subsistencia_ARCILLA,subsistencia_ARENAS_DE_RIO,subsistencia_ESMERALDAS,subsistencia_GRAVAS_DE_RIO,subsistencia_METALES_PRECIOSOS,subsistencia_MORRALLA,subsistencia_PIEDRAS_PRECIOSAS,subsistencia_PIEDRAS_SEMIPRECIOSAS,MPIO,Poblacion,...,subsistencia_PIEDRAS_PRECIOSAS_x10khab,subsistencia_PIEDRAS_SEMIPRECIOSAS_x10khab,LOG_subsistencia_ARCILLA_x10khab,LOG_subsistencia_ARENAS_DE_RIO_x10khab,LOG_subsistencia_ESMERALDAS_x10khab,LOG_subsistencia_GRAVAS_DE_RIO_x10khab,LOG_subsistencia_METALES_PRECIOSOS_x10khab,LOG_subsistencia_MORRALLA_x10khab,LOG_subsistencia_PIEDRAS_PRECIOSAS_x10khab,LOG_subsistencia_PIEDRAS_SEMIPRECIOSAS_x10khab
0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,05001,"2,526,795.00",...,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,05002,"21,126.00",...,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
# Dar estructura a los datos para acoplarse al PANEL
df_mineros_subsitencia['Año'] = 2026 # No tengo información de los años. registro los datos como observaciones de 2026

variables_y = ['LOG_subsistencia_ARCILLA_x10khab',
               'LOG_subsistencia_ARENAS_DE_RIO_x10khab',
               'LOG_subsistencia_ESMERALDAS_x10khab',
               'LOG_subsistencia_GRAVAS_DE_RIO_x10khab',
               'LOG_subsistencia_METALES_PRECIOSOS_x10khab',
               'LOG_subsistencia_MORRALLA_x10khab',
               'LOG_subsistencia_PIEDRAS_PRECIOSAS_x10khab',
               'LOG_subsistencia_PIEDRAS_SEMIPRECIOSAS_x10khab'
]
panel_mineros_subsistencia = df_mineros_subsitencia.melt(id_vars=['Código Dane municipio', 'Año'], 
                                                        value_vars=variables_y,
                                                        value_name='valor')
panel_mineros_subsistencia['auxiliar'] = panel_mineros_subsistencia['variable'].str.strip('LOG_subsistencia') # la variable auxiliar informa el tipo de mineral
panel_mineros_subsistencia['tipo_variable'] = 'Instrumento - Mineros subsistencia'


#-------------- Agregar al PANEL
panel_agregar = panel_mineros_subsistencia.copy()

# # Mostrar datos
print(f"\n{'*'*50} Variable a agregar")
display(panel_agregar.head(2))
print(f"\n{'*'*50}")
display(panel_agregar.info()) # Revisar estructura de los datos


# Agregar en estructura PANEL
print(f"\n{'*'*50} PANEL ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# # # Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} PANEL DESPUÉS de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})


************************************************** Variable a agregar


,Código Dane municipio,Año,variable,valor,auxiliar,tipo_variable
0,05001,2026,LOG_subsistencia_ARCILLA_x10khab,NaN,ARCILLA_x10kh,Instrumento - Mineros subsistencia
1,05002,2026,LOG_subsistencia_ARCILLA_x10khab,NaN,ARCILLA_x10kh,Instrumento - Mineros subsistencia



**************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8984 entries, 0 to 8983
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  8984 non-null   object 
 1   Año                    8984 non-null   int64  
 2   variable               8984 non-null   object 
 3   valor                  799 non-null    float64
 4   auxiliar               8984 non-null   object 
 5   tipo_variable          8984 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 421.3+ KB


None


************************************************** PANEL ANTES de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 278670 entries, 0 to 1121
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  278670 non-null  object 
 1   Año                    278670 non-null  int64  
 2   variable               278670 non-null  object 
 3   valor                  278670 non-null  float64
 4   auxiliar               278670 non-null  object 
 5   tipo_variable          278670 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 14.9+ MB


None


************************************************** PANEL DESPUÉS de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 287654 entries, 0 to 8983
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  287654 non-null  object 
 1   Año                    287654 non-null  int64  
 2   variable               287654 non-null  object 
 3   valor                  279469 non-null  float64
 4   auxiliar               287654 non-null  object 
 5   tipo_variable          287654 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 15.4+ MB


None

variable   Año        \
                                       nunique   min   max   
tipo_variable                                                
Control - Otros cultivos                    48  2019  2024   
Instrumento - Mineros subsistencia           8  2026  2026   
Instrumento - area titulo minero             2  1989  2021   
Instrumento - potencial geológico oro        1  2026  2026   
Resultado                                   12  2019  2024   
Tratamiento - producción regalías           76  2012  2026   

                                      Código Dane municipio   valor  
                                                    nunique   count  
tipo_variable                                                        
Control - Otros cultivos                               1102  212868  
Instrumento - Mineros subsistencia                     1123     799  
Instrumento - area titulo minero                        124   16368  
Instrumento - potencial geológico oro                  1122    1122  
Resultado                                               744   25674  
Tratamiento - producción regalías                       881   22638

## Controles
- Insumos agricolas (HECHO)
- Insumos produccion (HECHO)
- Elevación (PENDIENTE)
- Temperatura (PENDIENTE)
- Humedad (PENDIENTE)
- Precipitación (PENDIENTE)

### Insumos agrícolas

In [58]:
# Arreglar nombre de las columnas
display(panel_insumos.head(2)) # Mostrar datos para elegir columnas columnas y corregir nombres
panel_agregar = panel_insumos[['Código municipio', 'Año ', 'variable', 'valor', 'auxiliar', 'tipo_variable']]
panel_agregar = panel_agregar.rename(columns={'Código municipio':'Código Dane municipio',
                                             'Año ': 'Año'})

# Mostrar datos
print(f"\n{'*'*20}")
display(panel_agregar.head(2))
print(f"\n{'*'*20}")
display(panel_agregar.info()) # Revisar estructura de los datos

# Agregar en estructura PANEL
print(f"\n{'*'*50} ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} DESPUÉS de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})

,Código municipio,Año,tipo_insumo,Nombre del producto,u_salida,valor,variable,auxiliar,tipo_variable
0,05001,2021,"Coadyuvantes, molusquicidas, reguladores fisio...",Agrotin Sl,litro,"20,642.47",precio_Agrotin Sl_COP/litro,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
1,05001,2021,"Coadyuvantes, molusquicidas, reguladores fisio...",Baboxa Sb,kilogramo,"28,175.00",precio_Baboxa Sb_COP/kilogramo,"Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas



********************


,Código Dane municipio,Año,variable,valor,auxiliar,tipo_variable
0,05001,2021,precio_Agrotin Sl_COP/litro,"20,642.47","Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas
1,05001,2021,precio_Baboxa Sb_COP/kilogramo,"28,175.00","Coadyuvantes, molusquicidas, reguladores fisio...",Control - Insumos agrícolas



********************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53324 entries, 0 to 53323
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  53324 non-null  object 
 1   Año                    53324 non-null  int64  
 2   variable               53324 non-null  object 
 3   valor                  53324 non-null  float64
 4   auxiliar               53324 non-null  object 
 5   tipo_variable          53324 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 2.4+ MB


None

...

None


************************************************** DESPUÉS de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 340978 entries, 0 to 53323
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  340978 non-null  object 
 1   Año                    340978 non-null  int64  
 2   variable               340978 non-null  object 
 3   valor                  332793 non-null  float64
 4   auxiliar               340978 non-null  object 
 5   tipo_variable          340978 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 18.2+ MB


None


**************************************************
Resumen del PANEL
**************************************************


variable   Año        \
                                       nunique   min   max   
tipo_variable                                                
Control - Insumos agrícolas                770  2021  2026   
Control - Otros cultivos                    48  2019  2024   
Instrumento - Mineros subsistencia           8  2026  2026   
Instrumento - area titulo minero             2  1989  2021   
Instrumento - potencial geológico oro        1  2026  2026   
Resultado                                   12  2019  2024   
Tratamiento - producción regalías           76  2012  2026   

                                      Código Dane municipio   valor  
                                                    nunique   count  
tipo_variable                                                        
Control - Insumos agrícolas                             190   53324  
Control - Otros cultivos                               1102  212868  
Instrumento - Mineros subsistencia                     1123     799  
Instrumento - area titulo minero                        124   16368  
Instrumento - potencial geológico oro                  1122    1122  
Resultado                                               744   25674  
Tratamiento - producción regalías                       881   22638

### Insumos agropecuarios

In [59]:
panel_seleccion_insumos_produccion.columns

Index(['Año', 'Código_municipio', 'insumo', 'tipo_insumo', 'Presentacion',
       'Precio_promedio', 'variable', 'valor', 'auxiliar', 'tipo_variable'],
      dtype='object')

In [60]:
# Arreglar nombre de las columnas
display(panel_seleccion_insumos_produccion.head(2)) # Mostrar datos para elegir columnas columnas y corregir nombres
panel_agregar = panel_seleccion_insumos_produccion[['Código_municipio', 'Año', 'variable', 'valor', 'auxiliar', 'tipo_variable']]
panel_agregar = panel_agregar.rename(columns={'Código_municipio':'Código Dane municipio'})

# # Mostrar datos
print(f"\n{'*'*20}")
display(panel_agregar.head(2))
print(f"\n{'*'*20}")
display(panel_agregar.info()) # Revisar estructura de los datos

# Agregar en estructura PANEL
print(f"\n{'*'*50} ANTES de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

# Incorporar nuevas variables
PANEL = pd.concat([PANEL, panel_agregar])
print(f"\n{'*'*50} DESPUÉS de nuevas variables")
display(PANEL.info()) # Revisar estructura de los datos

print(f"\n{'*'*50}\nResumen del PANEL\n{'*'*50}")
PANEL.groupby('tipo_variable').agg({'variable':'nunique',
                                    'Año':['min', 'max'],
                                    'Código Dane municipio':'nunique',
                                    'valor':'count'})

,Año,Código_municipio,insumo,tipo_insumo,Presentacion,Precio_promedio,variable,valor,auxiliar,tipo_variable
0,2021,05001,Bovino Cebú macho 13-18 meses,Especies productivas,kilogramo,"6,786.25",Bovino Cebú macho 13-18 meses_kilogramo,"6,786.25",Especies productivas,Control - insumos agropecuarios
1,2021,05001,Bovino Cebú macho 19-24 meses,Especies productivas,kilogramo,"5,750.00",Bovino Cebú macho 19-24 meses_kilogramo,"5,750.00",Especies productivas,Control - insumos agropecuarios



********************


,Código Dane municipio,Año,variable,valor,auxiliar,tipo_variable
0,05001,2021,Bovino Cebú macho 13-18 meses_kilogramo,"6,786.25",Especies productivas,Control - insumos agropecuarios
1,05001,2021,Bovino Cebú macho 19-24 meses_kilogramo,"5,750.00",Especies productivas,Control - insumos agropecuarios



********************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4876 entries, 0 to 4875
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Código Dane municipio  4876 non-null   object 
 1   Año                    4876 non-null   int64  
 2   variable               4876 non-null   object 
 3   valor                  4876 non-null   float64
 4   auxiliar               4876 non-null   object 
 5   tipo_variable          4876 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 228.7+ KB


None

...

None


************************************************** DESPUÉS de nuevas variables
<class 'pandas.core.frame.DataFrame'>
Index: 345854 entries, 0 to 4875
Data columns (total 6 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Código Dane municipio  345854 non-null  object 
 1   Año                    345854 non-null  int64  
 2   variable               345854 non-null  object 
 3   valor                  337669 non-null  float64
 4   auxiliar               345854 non-null  object 
 5   tipo_variable          345854 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 18.5+ MB


None


**************************************************
Resumen del PANEL
**************************************************


variable   Año        \
                                       nunique   min   max   
tipo_variable                                                
Control - Insumos agrícolas                770  2021  2026   
Control - Otros cultivos                    48  2019  2024   
Control - insumos agropecuarios            103  2021  2026   
Instrumento - Mineros subsistencia           8  2026  2026   
Instrumento - area titulo minero             2  1989  2021   
Instrumento - potencial geológico oro        1  2026  2026   
Resultado                                   12  2019  2024   
Tratamiento - producción regalías           76  2012  2026   

                                      Código Dane municipio   valor  
                                                    nunique   count  
tipo_variable                                                        
Control - Insumos agrícolas                             190   53324  
Control - Otros cultivos                               1102  212868  
Control - insumos agropecuarios                         186    4876  
Instrumento - Mineros subsistencia                     1123     799  
Instrumento - area titulo minero                        124   16368  
Instrumento - potencial geológico oro                  1122    1122  
Resultado                                               744   25674  
Tratamiento - producción regalías                       881   22638

# Agregar variables a Panel existente

## Cargar version de panel existente

In [33]:
# Cargar última versión del panel
panel_anterior = pd.read_parquet('./outputs/PANEL_2026_04_21.parquet')

## Agregar Instrumento distancia a Au

In [34]:
# cargar panel de distancia al oro
panel_proximidad_oro = pd.read_parquet('../9_estimar_distancia_a_oro/outputs/distancia_proximidad_Au.parquet')

# Completar estructura del panel para ser compatible con los demás datos
panel_proximidad_oro = panel_proximidad_oro.rename(columns={
    'mpio_cdpmp': 'Código Dane municipio',
    'value': 'valor'})
panel_proximidad_oro['Año'] = 2026
panel_proximidad_oro['tipo_variable'] = 'Instrumento - Distancia al oro'
panel_proximidad_oro['Auxiliar'] = '' # Por ahora, vacío

## Nuevo panel

In [35]:
# Unir panel anterior con nuevos datos
nuevo_PANEL = pd.concat([panel_anterior, panel_proximidad_oro])

# Exportar datos

In [36]:
ruta_panel = './outputs/PANEL_2026_04_24'
nuevo_PANEL.to_parquet(ruta_panel + '.parquet')
nuevo_PANEL.to_excel(ruta_panel + '.xlsx')